In [1]:
import pandas as pd 
import json

with open('drebin_top_features.json') as f:
    drebin_features = json.load(f)
    
with open('malgenome_top_features.json') as f:
    malgenome_features = json.load(f)


In [2]:
# union of drebin and malgenome features
selected_features = list(set(drebin_features).union(set(malgenome_features)))
print(f"Total Unified Features: {len(selected_features)}")

Total Unified Features: 135


In [9]:
# loading both datasets
drebin_df = pd.read_csv('shuffled_apk_dataset.csv',low_memory=False)
malgenome_df = pd.read_csv('malgenome-215_shuffled.csv')

In [14]:
label_map = {'B': 0, 'S': 1}
drebin_df['class'] = drebin_df['class'].map(label_map)
malgenome_df['class'] = malgenome_df['class'].map(label_map)

In [15]:
# List of problematic row indices
bad_indices = [832, 1257, 4205, 7865, 14525]

# Drop from both X and y to keep them aligned
drebin_df = drebin_df.drop(index=bad_indices)
# Reset indices for cleanliness
drebin_df = drebin_df.reset_index(drop=True)

print("✅ Dropped rows with non-numeric values.")


✅ Dropped rows with non-numeric values.


In [20]:
# dropping rows with missing labels
drebin_df = drebin_df.dropna(subset=['class'])
malgenome_df = malgenome_df.dropna(subset=['class'])

In [21]:
print("drebin shape:",drebin_df.shape)
print("malgenome shape:" , malgenome_df.shape)



drebin shape: (15026, 218)
malgenome shape: (3799, 218)


In [18]:
# filling missing values with 0
for feature in selected_features:
    if feature not in drebin_df.columns:
        drebin_df[feature] = 0
    if feature not in malgenome_df.columns:
        malgenome_df[feature] =0
        

In [27]:
# subset to select + class
drebin_df = drebin_df[selected_features + ['class']]
malgenome_df = malgenome_df[selected_features + ['class']]

In [28]:
# ensuring all feartures are numeric
drebin_df[selected_features] = drebin_df[selected_features].fillna(0).astype(int)
malgenome_df[selected_features] = malgenome_df[selected_features].fillna(0).astype(int)

In [29]:
# combning both datasets
combined_df = pd.concat([drebin_df, malgenome_df], ignore_index=True)
# shuffling the combined dataset
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)
print("✅ Combined dataset shape:", combined_df.shape)

✅ Combined dataset shape: (18825, 136)


In [30]:
combined_df.head(10)

,SEND_SMS,RECEIVE_SMS,HttpPost.init,android.intent.action.PACKAGE_REMOVED,BLUETOOTH,android.intent.action.BOOT_COMPLETED,READ_PHONE_STATE,android.intent.action.SEND,android.intent.action.SEND_MULTIPLE,.system.app,...,IBinder,Ljava.lang.Class.cast,Ljava.lang.Class.getMethod,READ_PROFILE,HttpUriRequest,android.os.IBinder,android.content.pm.PackageInfo,Binder,READ_EXTERNAL_STORAGE,class
0,1,1,0,0,0,1,1,0,0,0,...,1,0,1,0,0,1,0,1,0,1
1,0,0,1,0,0,1,1,0,0,0,...,1,0,1,0,1,1,0,1,0,1
2,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,1,0,1,0,0
3,0,0,1,0,0,0,1,0,0,0,...,1,1,1,0,1,1,1,1,0,0
4,0,0,1,0,0,0,0,0,0,0,...,1,1,1,0,1,1,1,1,0,0
5,1,0,0,0,0,0,1,0,0,0,...,0,0,1,0,0,0,0,0,0,1
6,0,0,1,0,0,0,1,0,0,0,...,1,1,1,0,1,1,1,1,1,0
7,0,0,1,0,1,0,0,0,0,0,...,1,1,1,0,1,1,1,1,1,0
8,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,1,0,0,0
9,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,1,1,1,0,0


In [31]:
# saving the combined dataset
combined_df.to_csv('combined_dataset.csv', index=False)
print("✅ Combined dataset saved as 'combined_dataset.csv'")
# saving the selected features
with open('selected_features.json', 'w') as f:
    json.dump(selected_features, f)
print("✅ Selected features saved as 'selected_features.json'")

✅ Combined dataset saved as 'combined_dataset.csv'
✅ Selected features saved as 'selected_features.json'
